## Load useful libraries

In [16]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import cosine

In [2]:
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
import pyspark.sql.functions as F
from pyspark.sql import Window
from pyspark.sql.types import FloatType

## User settings

In [42]:
output_directory = 'output'
sampling_rate = 22050
hop_length = 512 * 100
spark_memory = '70G'
percentile_cutoff = 0.8
approx_quantile_precision = 0.05

# location of tracks to analyze for building the track feature database
path_library_parquet = '../database/music/output/playlist.parquet'

## Initialize Spark session

In [4]:
conf = (
    SparkConf()
    .setAppName('AnalyzedTrackLibrary')
    .set('spark.executor.memory', spark_memory)
    .set('spark.driver.memory', spark_memory)
    .set('spark.driver.maxResultSize', spark_memory)
)

spark = SparkSession.builder.config(conf = conf).getOrCreate()

26/03/19 17:41:59 WARN Utils: Your hostname, emily-MS-7B96 resolves to a loopback address: 127.0.1.1; using 192.168.1.99 instead (on interface eno1)
26/03/19 17:41:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/19 17:41:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Load show and track library data

In [7]:
path_show_output = output_directory + '/show_vectors_spark_sr_' + str(sampling_rate) + '_hl_' + str(hop_length) + '.parquet'
sdf_show = spark.read.parquet(path_show_output)

In [8]:
path_library_output = output_directory + '/library_vectors_spark_sr_' + str(sampling_rate) + '_hl_' + str(hop_length) + '.parquet'
sdf_library = spark.read.parquet(path_library_output)

## Join the show and track library dataframes

In [20]:
sdf_cross_joined = (
    sdf_show
    .crossJoin(sdf_library)
    .orderBy('time_step', 'id')
)

In [21]:
sdf_cross_joined.show(3)

+---------+--------------------+--------------------+---+
|time_step|          array_show|       array_library| id|
+---------+--------------------+--------------------+---+
|        0|[0.29534608125686...|[0.28707078099250...| 28|
|        0|[0.29534608125686...|[0.29539421200752...| 28|
|        0|[0.29534608125686...|[0.28956532478332...| 28|
+---------+--------------------+--------------------+---+
only showing top 3 rows



## Define a function for computing cosine similarity

In [22]:
@F.udf(returnType=FloatType())
def compute_cosine_similarity(vector1, vector2):
    cosine_dist = cosine(np.array(vector1), np.array(vector2))
    similarity_score = 1 - cosine_dist
    return float(similarity_score)

## Compute cosine similarity

In [23]:
sdf_cross_joined = (
    sdf_cross_joined
    .withColumn('cosine_similarity', compute_cosine_similarity(F.col('array_show'), F.col('array_library')))
    .drop('array_show', 'array_library')
    .orderBy('time_step', 'id')
)

In [24]:
sdf_cross_joined.show(5)

+---------+---+-----------------+
|time_step| id|cosine_similarity|
+---------+---+-----------------+
|        0| 28|       0.68394727|
|        0| 28|        0.6860955|
|        0| 28|          0.68217|
|        0| 28|        0.6945333|
|        0| 28|        0.6886212|
+---------+---+-----------------+
only showing top 5 rows



## Reduce dataset size by percentile cutoff

In [26]:
sdf_cross_joined.repartition(100)  # I just made this number up, there is no specific rationale for it.

DataFrame[time_step: bigint, id: bigint, cosine_similarity: float]

In [27]:
sdf_cross_joined.count()

146762760

In [28]:
similarity_quantile_cutoff = sdf_cross_joined.approxQuantile('cosine_similarity', [percentile_cutoff], approx_quantile_precision)

In [29]:
similarity_quantile_cutoff

[0.8558716177940369]

In [30]:
sdf_cross_joined = (
    sdf_cross_joined
    .where(F.col('cosine_similarity') >= F.lit(similarity_quantile_cutoff[0]))
    .orderBy('time_step', 'id')
)

In [31]:
sdf_cross_joined.count()

26/03/19 18:17:23 WARN ExtractPythonUDFFromJoinCondition: The join condition:(compute_cosine_similarity(array_show#5, array_library#8)#105 >= 0.8558716) of the join plan contains PythonUDF only, it will be moved out and the join plan will be turned to cross join.


30377469

## Aggregate by (timestamp, song_id)

We retain the maximum cosine similarity per (timestamp / song ID) pair:

In [33]:
sdf_cross_joined.repartition('time_step', 'id')

sdf_agg = (
    sdf_cross_joined
    .groupBy('time_step', 'id')
    .agg(
        F.max('cosine_similarity').alias('cosine_similarity'),
    )
    .orderBy('time_step', F.desc('cosine_similarity'))
)

In [34]:
sdf_agg.show(5)

26/03/19 18:28:00 WARN ExtractPythonUDFFromJoinCondition: The join condition:(compute_cosine_similarity(array_show#5, array_library#8)#105 >= 0.8558716) of the join plan contains PythonUDF only, it will be moved out and the join plan will be turned to cross join.


+---------+----+-----------------+
|time_step|  id|cosine_similarity|
+---------+----+-----------------+
|        0|1176|       0.99054056|
|        0|2173|        0.9294219|
|        0|1280|        0.9153691|
|        0|5241|       0.91474766|
|        0|4375|       0.89411944|
+---------+----+-----------------+
only showing top 5 rows



In [36]:
path_agg_output = output_directory + '/agg_spark_sr_' + str(sampling_rate) + '_hl_' + str(hop_length) + '.parquet'
sdf_library.write.mode('overwrite').parquet(path_agg_output)

## Record the rank per timestamp¶

In [38]:
sdf_agg.repartition('time_step')

DataFrame[time_step: bigint, id: bigint, cosine_similarity: float]

In [40]:
window_spec = Window.partitionBy('time_step').orderBy(F.desc('cosine_similarity'))

sdf_agg_ranked = (
    sdf_agg
    .orderBy(F.asc('time_step'), F.desc('cosine_similarity'))
    .withColumn('rank', F.row_number().over(window_spec))
)

In [41]:
sdf_agg_ranked.show(5)

26/03/19 18:42:28 WARN ExtractPythonUDFFromJoinCondition: The join condition:(compute_cosine_similarity(array_show#5, array_library#8)#105 >= 0.8558716) of the join plan contains PythonUDF only, it will be moved out and the join plan will be turned to cross join.


+---------+----+-----------------+----+
|time_step|  id|cosine_similarity|rank|
+---------+----+-----------------+----+
|        0|1176|       0.99054056|   1|
|        0|2173|        0.9294219|   2|
|        0|1280|        0.9153691|   3|
|        0|5241|       0.91474766|   4|
|        0|4375|       0.89411944|   5|
+---------+----+-----------------+----+
only showing top 5 rows



## Keep only the top-ranked rows per time step

In [43]:
sdf_agg_ranked = sdf_agg_ranked.where(F.col('rank') <= 1)

In [44]:
sdf_agg_ranked.show(5)

26/03/19 18:55:05 WARN ExtractPythonUDFFromJoinCondition: The join condition:(compute_cosine_similarity(array_show#5, array_library#8)#105 >= 0.8558716) of the join plan contains PythonUDF only, it will be moved out and the join plan will be turned to cross join.


+---------+----+-----------------+----+
|time_step|  id|cosine_similarity|rank|
+---------+----+-----------------+----+
|        0|1176|       0.99054056|   1|
|        1|1176|       0.99080956|   1|
|        2|1176|        0.9910114|   1|
|        3|1176|       0.99117124|   1|
|        4|1176|       0.99135906|   1|
+---------+----+-----------------+----+
only showing top 5 rows



In [45]:
sdf_agg_ranked = sdf_agg_ranked.drop('rank')

In [46]:
sdf_agg_ranked.show(5)

26/03/19 19:08:07 WARN ExtractPythonUDFFromJoinCondition: The join condition:(compute_cosine_similarity(array_show#5, array_library#8)#105 >= 0.8558716) of the join plan contains PythonUDF only, it will be moved out and the join plan will be turned to cross join.


+---------+----+-----------------+
|time_step|  id|cosine_similarity|
+---------+----+-----------------+
|        0|1176|       0.99054056|
|        1|1176|       0.99080956|
|        2|1176|        0.9910114|
|        3|1176|       0.99117124|
|        4|1176|       0.99135906|
+---------+----+-----------------+
only showing top 5 rows



In [49]:
sdf_agg_ranked = sdf_agg_ranked.drop('cosine_similarity')

In [50]:
sdf_agg_ranked.show(5)

26/03/19 19:20:28 WARN ExtractPythonUDFFromJoinCondition: The join condition:(compute_cosine_similarity(array_show#5, array_library#8)#105 >= 0.8558716) of the join plan contains PythonUDF only, it will be moved out and the join plan will be turned to cross join.


+---------+----+
|time_step|  id|
+---------+----+
|        0|1176|
|        1|1176|
|        2|1176|
|        3|1176|
|        4|1176|
+---------+----+
only showing top 5 rows



In [51]:
path_rank_output = output_directory + '/rank_spark_sr_' + str(sampling_rate) + '_hl_' + str(hop_length) + '.parquet'
sdf_agg_ranked.write.mode('overwrite').parquet(path_rank_output)

26/03/19 19:46:56 WARN ExtractPythonUDFFromJoinCondition: The join condition:(compute_cosine_similarity(array_show#5, array_library#8)#105 >= 0.8558716) of the join plan contains PythonUDF only, it will be moved out and the join plan will be turned to cross join.


In [89]:
sdf_ids = spark.read.parquet(path_rank_output)

In [90]:
sdf_ids = (
    sdf_ids
    .withColumn('timestamp', F.col('time_step') * (hop_length / sampling_rate))
)

In [91]:
sdf_ids.show(5)

+---------+----+------------------+
|time_step|  id|         timestamp|
+---------+----+------------------+
|        0|1176|               0.0|
|        1|1176|2.3219954648526078|
|        2|1176|4.6439909297052155|
|        3|1176| 6.965986394557824|
|        4|1176| 9.287981859410431|
+---------+----+------------------+
only showing top 5 rows



In [92]:
sdf_ids.repartition('id')

DataFrame[time_step: bigint, id: bigint, timestamp: double]

In [93]:
sdf_ids_agg = (
    sdf_ids
    .groupBy('id')
    .agg(
        #F.min('timestamp').alias('start'),
        #F.max('timestamp').alias('end'),
        F.percentile_approx('timestamp', 0.25).alias('start'),
        F.percentile_approx('timestamp', 0.75).alias('end'),
    )
    .orderBy('start')
)

In [94]:
sdf_ids_agg.show(10)

+----+------------------+------------------+
|  id|             start|               end|
+----+------------------+------------------+
|1176|23.219954648526077| 74.30385487528345|
|3785|136.99773242630386| 208.9795918367347|
|3865|301.85941043083903| 424.9251700680272|
|3755| 494.5850340136055|11482.267573696145|
| 506| 531.7369614512472| 596.7528344671202|
|4334| 633.9047619047619| 638.5487528344671|
| 161| 659.4467120181406| 691.9546485260771|
|2307| 745.3605442176871| 819.6643990929706|
|3181|  893.968253968254| 945.0521541950113|
|3199| 991.4920634920635|1030.9659863945578|
+----+------------------+------------------+
only showing top 10 rows



## Load the track titles and artists

In [95]:
sdf_library_names = (
    spark
    .createDataFrame(pd.read_parquet(path_library_parquet))
    .drop('path')
    .orderBy('id')
)

In [96]:
sdf_library_names.show(5)

+---+--------------------+----------------+
| id|                name|          artist|
+---+--------------------+----------------+
| 28|   On My Way to Hell|Połoz & Tinnitus|
| 32|        Militia Love|             SMP|
| 35|      Slowly Melting|       Nomeansno|
| 42|Blue Monday (Elec...|    Armada Tribe|
| 53|Undisclosed Desir...|            Muse|
+---+--------------------+----------------+
only showing top 5 rows



In [97]:
sdf_named = (
    sdf_ids_agg.join(sdf_library_names, on = 'id', how = 'left')
    .withColumn('IRQ', F.col('end') - F.col('start'))
    .orderBy('start')
    .drop('id')
)

In [98]:
sdf_named.show(10)

+------------------+------------------+--------------------+--------------------+------------------+
|             start|               end|                name|              artist|               IRQ|
+------------------+------------------+--------------------+--------------------+------------------+
|23.219954648526077| 74.30385487528345| Shock To The System|          Billy Idol| 51.08390022675737|
|136.99773242630386| 208.9795918367347|N.W.O. (Re-Record...|            Ministry| 71.98185941043084|
|301.85941043083903| 424.9251700680272|Marcha Funebre [E...|    Divina Blasfemia| 123.0657596371882|
| 494.5850340136055|11482.267573696145|Time Will Destroy...|             Garbage|10987.682539682539|
| 531.7369614512472| 596.7528344671202|           Hyperpunk|       Head Splitter| 65.01587301587301|
| 633.9047619047619| 638.5487528344671|Paint it Black (M...|Danny Darko, Juli...| 4.643990929705183|
| 659.4467120181406| 691.9546485260771|Bitterer Als Der Tod|       Boris Mikulic|32.5079365

In [99]:
irq_min = 75. / 2.
irq_max = 60. * 10

In [100]:
sdf_named_cut_min_time_diff = (
    sdf_named
    .where(F.col('IRQ') >= irq_min)
    .where(F.col('IRQ') < irq_max)
    .orderBy('start')
)

In [101]:
sdf_named_cut_min_time_diff.show(300)

+------------------+------------------+--------------------+--------------------+------------------+
|             start|               end|                name|              artist|               IRQ|
+------------------+------------------+--------------------+--------------------+------------------+
|23.219954648526077| 74.30385487528345| Shock To The System|          Billy Idol| 51.08390022675737|
|136.99773242630386| 208.9795918367347|N.W.O. (Re-Record...|            Ministry| 71.98185941043084|
|301.85941043083903| 424.9251700680272|Marcha Funebre [E...|    Divina Blasfemia| 123.0657596371882|
| 531.7369614512472| 596.7528344671202|           Hyperpunk|       Head Splitter| 65.01587301587301|
| 745.3605442176871| 819.6643990929706|          Wonderland|       Natalia Kills| 74.30385487528349|
|  893.968253968254| 945.0521541950113|          Underwater|            ALTIMAIT| 51.08390022675735|
| 991.4920634920635|1030.9659863945578|She's Got Little ...|john crozier harr...| 39.473922